# Time Scale Modification

In this homework you will implement several algorithms for time scale modification -- changing the playback speed of audio without changing the pitch.

As we are drawing near to the final project, I will be providing very little hand holding -- I will give you broad tasks to accomplish without providing answers. It is up to you to verify that your code is working!

Please note that an additional 10 points (10%) of your grade will be on the readability and decomposition of your code, as well as the clarity of your writing.  You should explain your thought process clearly and support your claims with appropriate plots.  Please submit your .ipynb and your generated audio recording (for the last part) as a .zip file.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa as lb
import IPython.display as ipd
from scipy.signal import medfilt

### Approach 1: Overlap-Add (OLA) [20 points]

The first approach you will implement is the overlap-add (OLA) method.  In this section, you should do the following:
- implement a function called tsm_overlap_add which takes three arguments: an input signal x, the time stretch factor alpha, and the frame length L.  It should return the time-stretched signal.  Your function should use default values of alpha=1 and L=220, and it should use a synthesis hop size that is half the value of L.
- apply your function to time stretch the beatboxing audio sample (with 22050 Hz sampling rate) by a factor of 1.25 (slowed down) and 0.75 (sped up).  Embed the resulting audio files in the notebook.
- apply your function to time stretch the choir audio sample by a factor of 1.25 (slowed down) and 0.75 (sped up).  Embed the resulting audio file in the notebook.
- Comment on the quality of the OLA algorithm when applied to the beatbox and choir audio samples.

In [ ]:
def tsm_overlap_add (x, alpha = 1, L = 220):
    Hs = L//2
    Ha = Hs//alpha
    Hs = int(Hs)
    Ha = int(Ha)
    Xm = get_Xm(x, L, Ha)
    Ymn = get_Ym(Xm, x)
    yn = get_Time_Stretched_Signal(Ymn, L, x, alpha, Hs)
    return yn

In [ ]:
def get_Xm(x, L, hopSize):
    """
    Inputs
      - x: an array carrying the audio data in order of time aquired
      - L: The length of each frame
      - hopsize: The distance between frames 
        
    Outputs
      - Xm: 2D array of analysis frames
    """
    numFrames = 1+(len(x)-L)//hopSize
    numFrames = int(numFrames)
    Xm = np.zeros((numFrames,L))

    startIndex = 0
    print(L)
    for i in range(numFrames):
        xn = x[startIndex:startIndex+L]
        Xm[i] = xn
        startIndex += hopSize

    return Xm
        

In [ ]:
def get_Ym(xM, x):
    numFrames = len(xM)
    ymN= np.zeros((numFrames,len(xM[0])))
    for n in range(numFrames):
        wN = 0.5*(1-np.cos((2*np.pi)/len(x)))
        ymN[n]= wN * xM[n]

    return ymN

In [ ]:
def get_Time_Stretched_Signal(ymN, L, x, alpha, hopSize):
    numFrames = len(ymN)
    yn = np.zeros(int(alpha*len(x)))
    yn = np.zeros((L // 2)*(numFrames -1 ) + L)
    beg = 0
    end = len(ymN[0])

    for y in ymN:
        yn[beg:end] += y
        beg += hopSize
        end += hopSize
    return yn

In [ ]:
y, sr = lb.load('beatbox.wav')
ipd.Audio(data=y, rate=sr)

In [ ]:
# print(y)
y, sr = lb.load('choir.wav')
# xM = get_Xm(y, 220, 110)
# print(xM)
# ymN = get_Ym(xM, y)
# get_Time_Stretched_Signal(ymN, 220, y, 1, 110)
#For alpha = 1.25
finalY = tsm_overlap_add (y, alpha = 1.25, L = 220)
ipd.Audio(data=finalY, rate=sr)
#For alpha = 0.75
# finalY = tsm_overlap_add (y, alpha = 0.75, L = 220)
# ipd.Audio(data=finalY, rate=sr)

### Approach 2: Phase Vocoder [45 points]

The second approach you will implement is the phase vocoder method.  In this section, you should do the following:
- implement a function called tsm_phase_vocoder which takes three arguments: an input signal x, the time stretch factor alpha, the frame length L, and the sample rate sr.  It should return the time-stretched signal.  Your function should use default values of alpha=1, L=2048, and sr=22050, and it should use a synthesis hop size that is one-fourth the value of L.  For the STFT, you may use the librosa stft function with FFT size 2048.  However, you should implement the istft function yourself (do *not* use the librosa istft function or look at its source code).
- apply your function to time stretch the beatboxing audio sample (with 22050 Hz sampling rate) by a factor of 1.25 (slowed down) and 0.75 (sped up).  Embed the resulting audio files in the notebook.
- apply your function to time stretch the choir audio sample by a factor of 1.25 (slowed down) and 0.75 (sped up).  Embed the resulting audio files in the notebook.
- Comment on the quality of the phase vocoder algorithm when applied to the beatbox and choir audio samples.

In [ ]:
def tsm_phase_vocoder(x, alpha = 1, L = 2048, sr = 22050):
    Hs = int(L/4)

    #figure out what signal is
    signal = 0
    return signal


In [ ]:
def compute_STFT(x, L, Hs, alpha): 
    Ha = int(Hs/alpha)
    windowLength =  wN = 0.5*(1-np.cos((2*np.pi)/len(x)))
    return lb.stft(x, L, Ha, windowLength )


### Harmonic Percussive Separation [15 points]

In preparation for implementing the third time scale modification approach, you will implement harmonic percussive separation.  In this section, you should do the following:
- implement a function called harmonic_percussive_separation which takes five arguments: an input signal x, sample rate sr, FFT size fft_size, hop size hop_length, harmonic median filter half-length lh, and percussive median filter half-length lp.  It should return four outputs: the two time-domain signals xh and xp corresponding to the harmonic and percussive components, and the two corresponding modified STFTs Xh and Xp.  Your function should use default values sr=22050, fft_size=2048, hop_length=512, lh=6, and lp=6.  You may use the librosa stft function, but you should implement the rest yourself.  You should *not* use the librosa hpss function or look at its source code.  Make sure to reuse your istft function from above!
- superimpose the choir and beatboxing audio recordings together by taking the average of their time domain signals.  Process the mixture with your algorithm to estimate the harmonic and percussive components, and embed the two resulting audio recordings in the notebook.
- Compute the log-magnitude spectrogram of the estimated xh and xp signals, and visualize a short section of both specgtrograms.  Include the images in your notebook and comment on what you see.

### Approach 3: Hybrid Method [5 points]

The third approach you will implement is the hybrid method that combines both the overlap-add and phase vocoder methods. In this section, you should do the following:
- implement a function called tsm_hybrid which takes five arguments: an input signal x, time stretch factor alpha, and sample rate sr.  It should return the time-stretched signal y.  Your function should assume default values of alpha=1, sr=22050.  You should call the functions you implemented in earlier sections and use the corresponding default values.
- apply your function to time stretch the beatboxing audio sample by a factor of 1.25 (slowed down) and 0.75 (sped up). Embed the resulting audio files in the notebook.
- apply your function to time stretch the choir audio sample by a factor of 1.25 (slowed down) and 0.75 (sped up). Embed the resulting audio files in the notebook.
- Comment on the quality of the hybrid algorithm when applied to the beatbox and choir audio samples.

### Make something cool! [5 points]

Now use the functions you've implemented above to create a cool 10-15 second audio sample of any music you like.  Feel free to time stretch, pitch shift, and mix audio recordings to create something original.  You should include the following:
- code for generating the audio sample
- the audio file embedded into the notebook
- a brief explanation of what you did

Please also include your audio file in the zip file with your submission.